# Cross-species divergent splice sites, per timepoint (alignment-based)

A companion to [splice_cross_species_alignment_genome.ipynb](splice_cross_species_alignment_genome.ipynb).
Here **each timepoint is treated independently** (no averaging across timepoints): find sites
where — at *some* timepoint in `FOCUS_TISSUE` — splice usage is **high in human and low in the
other species** (mouse / rat / rabbit), the model predicts that condition well in every species
involved, and (ideally) the site is aligned & present in **all** species.

Shared loaders / metrics / MAF lift-over live in `alphagenome_pytorch.xspecies` (reused by both
notebooks); the whole-genome-alignment lift-over cache is shared and tissue-independent.

## Pipeline
1. **Human chr8 sites** — per-(site, timepoint) true/pred usage in `FOCUS_TISSUE`.
2. **Lift-over** — aligned coordinate in each other species (cached MAF lift-over; rat rn6→rn5).
3. **Other species** — per-timepoint true/pred usage at the aligned site.
4. **Select** — the best timepoint per site: human high, all present species low, every condition
   well predicted; ranked by (present-in-all-species, usage gap).

## Setup

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from alphagenome_pytorch import xspecies as sx
from alphagenome_pytorch.plotting.splicing import (
    plot_splice_site_predictions,
    TISSUE_COLORS,
    TISSUE_ORDER,
    SPECIES_SCI,
)

TPS = list(range(1, 16))  # developmental timepoints 1..15

plt.rcParams.update({"figure.dpi": 100, "font.size": 10})

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR   = "/home/elek/sds/sd17d003/Anamaria/alphagenome_genomicsxai/data"
USAGE_TMPL = os.path.join(DATA_DIR, "combined_usage_data_{species}.parquet")
MODEL      = "lora_32_human_mouse_rat_rabbit_opossum"
PRED_DIR   = os.path.join("/home/elek/sds/sd17d003/Anamaria/alphagenome_genomicsxai",
                          MODEL, "preds_intersect_protein_coding")

CHROMS     = ["1", "3", "4", "5", "8"]   # one or more chromosomes, e.g. ["5", "8"]
ASSEMBLY   = {"mouse": "mm10", "rat": "rn6", "rabbit": "oryCun2"}
OTHER_SPECIES = ["mouse", "rat", "rabbit"]
RAT_CHAIN  = "/home/elek/sds/sd17d003/Anamaria/genomes/chains/rn6ToRn5.over.chain.gz"

FOCUS_TISSUE   = "Brain"
ALIGN_TOL      = 5      # bp: snap an aligned coordinate to the nearest predicted site
MIN_OBS        = 5      # min observed timepoints to score a site
HIGH_HUMAN     = 0.60   # human mean usage >= this -> "high"
LOW_OTHER      = 0.40   # other-species mean usage <= this -> "low"
ERR_WELL       = 0.12   # Error allowed for "well predicted"
MIN_USAGE_GAP  = 0.20   # (human - other) mean-usage gap required

OUT_DIR = os.path.join(PRED_DIR, "human", "cross_species_usage")
os.makedirs(OUT_DIR, exist_ok=True)

# per-chromosome MAF / reference prefix / shared (tissue-independent) lift-over cache
def maf_path(c):       return f"/home/elek/sds/sd17d003/Anamaria/genomes/multiz100way/chr{c}.maf.gz"
def ref_prefix(c):     return f"hg38.chr{c}"
def liftover_cache(c): return os.path.join(PRED_DIR, "human", "cross_species_alignment",
                                           f"liftover_chr{c}.parquet")
CHROM_TAG = "_".join(CHROMS)

# thin wrappers binding config to the shared module
def load_predictions(sp, tissue, positions=None):
    p = sx.load_predictions(PRED_DIR, DATA_DIR, SPECIES_SCI, sp, tissue, positions)
    return sx.attach_strand(p, USAGE_TMPL, sp)   # add Strand -> site ids carry strand
def load_all_tissues(sp, positions):
    return sx.load_all_tissue_preds(PRED_DIR, DATA_DIR, SPECIES_SCI, sp, positions)

print("focus tissue:", FOCUS_TISSUE, "| chromosomes:", CHROMS, "| other species:", OTHER_SPECIES)
print(("OK  " if os.path.exists(RAT_CHAIN) else "MISS") + f"  rat chain: {RAT_CHAIN}")
for c in CHROMS:
    for lbl, pth in [(f"chr{c} MAF", maf_path(c)), (f"chr{c} liftover cache", liftover_cache(c))]:
        print(("OK  " if os.path.exists(pth) else "MISS") + f"  {lbl}: {pth}")

## 1 — Human chr8 sites: mean usage & accuracy (FOCUS_TISSUE)

Keep sites with **high** human usage that the model predicts well.

In [3]:
pred_h = load_predictions("human", FOCUS_TISSUE)
if pred_h is None:
    raise FileNotFoundError("No human predictions found")
pred_h = pred_h[pred_h["Chromosome"].isin(CHROMS)].copy()

# per-(site, timepoint) true & predicted usage — no averaging across timepoints
human_true, human_pred = sx.usage_wide(pred_h)
n_obs = np.isfinite(human_true.to_numpy()).sum(axis=1)
human_true = human_true[n_obs >= MIN_OBS]
human_pred = human_pred.reindex(human_true.index)
human_sites = list(human_true.index)
print(f"Human chr{CHROM_TAG} {FOCUS_TISSUE}: {len(human_sites):,} sites with >= {MIN_OBS} timepoints")
human_true.head()

Human chr1_3_4_5_8 Brain: 4,887 sites with >= 5 timepoints


Timepoint,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
site,,,,,,,,,,,,,,,
1:10007569,0.400,0.067,0.077,0.917,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.683,0.897,0.983,0.923
1:10007747,NaN,0.000,0.200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.970,1.000,1.000,0.950
1:10008172,NaN,0.000,0.188,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.963,0.969,0.875,0.950
1:10008273,NaN,0.133,0.143,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.955,1.000,0.905,0.833
1:100133314,0.571,0.500,0.579,0.911,0.667,0.75,0.718,0.769,0.85,0.921,0.85,0.500,0.769,0.700,NaN


## 2 — Lift over to other species (shared MAF alignment cache)

In [ ]:
# Per-chromosome MAF lift-over, concatenated. Each chromosome has its own cache.
all_hpos = pd.read_parquet(sx.pred_path(PRED_DIR, "human"),
                           columns=["Chromosome", "Position"]).drop_duplicates()
asm2sp = {a: s for s, a in ASSEMBLY.items()}
parts = []
for c in CHROMS:
    cache = liftover_cache(c)
    if os.path.exists(cache):
        part = pd.read_parquet(cache)
        print(f"chr{c}: loaded cache ({len(part):,} rows)")
    else:
        pos = sorted(all_hpos.loc[all_hpos["Chromosome"].astype(str) == c, "Position"]
                     .astype(int).unique().tolist())
        print(f"chr{c}: parsing MAF for {len(pos):,} human sites ...")
        res = sx.maf_liftover(maf_path(c), ref_prefix(c), pos, set(ASSEMBLY.values()))
        part = sx.liftover_to_frame(res, asm2sp, c)
        os.makedirs(os.path.dirname(cache), exist_ok=True)
        part.to_parquet(cache, index=False)
        print(f"chr{c}: saved -> {cache}")
    parts.append(part)
lift_df = pd.concat(parts, ignore_index=True)

# canonicalise human_site to chrom:pos:strand (upgrades legacy strand-less caches)
h_strand = sx.strand_series(USAGE_TMPL, "human")
lift_df["human_site"] = lift_df["human_site"].map(lambda s: sx.ensure_site_strand(s, h_strand))

# rat MAF coords are rn6; rat predictions are rn5 -> chain lift
if RAT_CHAIN and os.path.exists(RAT_CHAIN) and (lift_df["species"] == "rat").any():
    lift_df = sx.chain_lift(lift_df, "rat", RAT_CHAIN, new_assembly_label="rn6->rn5")

# keep only species / sites we analyse
lift_df = lift_df[lift_df["species"].isin(OTHER_SPECIES)
                  & lift_df["human_site"].isin(set(human_sites))].copy()
print("aligned rows for selected human sites:", lift_df["species"].value_counts().to_dict())
lift_df.head()

## 3 — Other species: aligned-site mean usage & accuracy

In [ ]:
records = {s: {} for s in human_sites}          # human_site -> {species: dict(aln_site, true, pred)}
for sp in OTHER_SPECIES:
    sub = lift_df[lift_df["species"] == sp]
    if sub.empty:
        print(f"{sp}: no alignments"); continue
    pos_expand = sorted({int(p) + d for p in sub["aln_pos"] for d in range(-ALIGN_TOL, ALIGN_TOL + 1)})
    pr = load_predictions(sp, FOCUS_TISSUE, positions=pos_expand)
    if pr is None:
        print(f"{sp}: predictions MISSING"); continue
    tw, pw = sx.usage_wide(pr)                      # per-(site, timepoint), conditions independent
    parsed = [sx.parse_site(s) for s in tw.index]
    chroms = np.array([q[0] for q in parsed]); posns = np.array([q[1] for q in parsed])
    site_by_cp = {(q[0], q[1]): s for q, s in zip(parsed, tw.index)}
    pos_by_chrom = {c: np.sort(posns[chroms == c]) for c in np.unique(chroms)}
    n = 0
    for _, r in sub.iterrows():
        chrom, apos = str(r["aln_chrom"]), int(r["aln_pos"])
        snapped = sx.nearest(pos_by_chrom.get(chrom, np.array([])), apos, ALIGN_TOL)
        if snapped is None:
            continue
        ssite = site_by_cp[(chrom, snapped)]        # full chrom:pos:strand id
        records[r["human_site"]][sp] = dict(
            aln_site=ssite, aln_dist=abs(snapped - apos),
            true=tw.loc[ssite].to_numpy(), pred=pw.loc[ssite].to_numpy())
        n += 1
    print(f"{sp}: {n:,} aligned sites matched to a predicted site")

## 4 — Qualifying timepoints: high in human, low in other species, well predicted

Scan each timepoint independently and keep **every** qualifying (site, timepoint) — a site may
appear at several timepoints. A timepoint qualifies when human usage `>= HIGH_HUMAN` (predicted
within `ERR_WELL`), every present other species is `<= LOW_OTHER` (predicted within `ERR_WELL`),
and the gap human − (nearest other) is `>= MIN_USAGE_GAP`. Rows are ranked by present-in-all-
species then gap.

In [6]:
TP = np.array(TPS)
rows = []
for hsite in human_sites:
    ht = human_true.loc[hsite].to_numpy(); hp = human_pred.loc[hsite].to_numpy()
    rec = records[hsite]
    for ti in range(len(TP)):
        if not np.isfinite(ht[ti]):
            continue
        present = [sp for sp in OTHER_SPECIES if sp in rec and np.isfinite(rec[sp]["true"][ti])]
        if not present:
            continue
        other_true = {sp: rec[sp]["true"][ti] for sp in present}
        gap = ht[ti] - max(other_true.values())          # human above the nearest other species
        human_ok  = ht[ti] >= HIGH_HUMAN and abs(ht[ti] - hp[ti]) <= ERR_WELL
        others_lo = all(v <= LOW_OTHER for v in other_true.values())
        others_ok = all(abs(rec[sp]["true"][ti] - rec[sp]["pred"][ti]) <= ERR_WELL for sp in present)
        if not (human_ok and others_lo and others_ok and gap >= MIN_USAGE_GAP):
            continue
        row = dict(site=hsite, timepoint=int(TP[ti]), gap=round(float(gap), 3),
                   n_species=len(present), all_species=(len(present) == len(OTHER_SPECIES)),
                   species=",".join(present),
                   human_true=round(float(ht[ti]), 3), human_pred=round(float(hp[ti]), 3))
        for sp in OTHER_SPECIES:
            row[f"{sp}_true"] = round(other_true[sp], 3) if sp in other_true else np.nan
        rows.append(row)

result = pd.DataFrame(rows)
if len(result):
    result = result.sort_values(["all_species", "gap"], ascending=[False, False]).reset_index(drop=True)
out_csv = os.path.join(OUT_DIR, f"usage_divergent_conditions_chr{CHROM_TAG}_{FOCUS_TISSUE.lower()}.csv")
result.to_csv(out_csv, index=False)
n_sites = result["site"].nunique() if len(result) else 0
n_all = int(result["all_species"].sum()) if len(result) else 0
print(f"{len(result):,} qualifying (site, timepoint) conditions across {n_sites:,} sites "
      f"({n_all:,} conditions present in all species)")
print(f"Saved -> {out_csv}")
result.head(20)

21 qualifying (site, timepoint) conditions across 16 sites (0 conditions present in all species)
Saved -> /home/elek/sds/sd17d003/Anamaria/alphagenome_genomicsxai/lora_32_human_mouse_rat_rabbit_opossum/preds_intersect_protein_coding/human/cross_species_usage/usage_divergent_conditions_chr1_3_4_5_8_brain.csv


,site,timepoint,gap,n_species,all_species,species,human_true,human_pred,mouse_true,rat_true,rabbit_true
0,5:140632979,5,0.977,1,False,mouse,0.977,0.909,0.000,NaN,NaN
1,3:113407040,5,0.897,1,False,rabbit,0.897,0.800,NaN,NaN,0.000
2,3:113396726,5,0.867,2,False,"mouse,rabbit",0.867,0.809,0.000,NaN,0.000
3,5:140632979,14,0.846,1,False,mouse,0.846,0.928,0.000,NaN,NaN
4,3:113407040,7,0.833,1,False,rabbit,0.833,0.808,NaN,NaN,0.000
5,3:113363531,7,0.829,1,False,rabbit,0.829,0.859,NaN,NaN,0.000
6,3:9751769,13,0.805,1,False,mouse,0.972,0.893,0.167,NaN,NaN
7,3:113358874,5,0.789,1,False,rabbit,0.789,0.728,NaN,NaN,0.000
8,3:113401738,5,0.769,1,False,rabbit,0.769,0.844,NaN,NaN,0.000
9,3:113344711,5,0.760,1,False,rabbit,0.760,0.653,NaN,NaN,0.000


## 4b — Filter funnel: which criterion removes the candidates

For every (site, timepoint) with finite human usage we tally, **independently**, how many
conditions fail each criterion, and the **cumulative** survivors when the criteria are applied
in order. This shows which threshold (`HIGH_HUMAN`, human accuracy, `LOW_OTHER`, other-species
accuracy, `MIN_USAGE_GAP`, presence) is the binding constraint.

In [7]:
from collections import Counter

TP = np.array(TPS)
fail = Counter()      # conditions failing this criterion (independent of others)
cum  = Counter()      # survivors after applying criteria cumulatively, in order
tot_conditions = 0
sites_with_finite = set()

for hsite in human_sites:
    ht = human_true.loc[hsite].to_numpy(); hp = human_pred.loc[hsite].to_numpy()
    rec = records[hsite]
    for ti in range(len(TP)):
        if not np.isfinite(ht[ti]):
            continue
        tot_conditions += 1
        sites_with_finite.add(hsite)

        present   = [sp for sp in OTHER_SPECIES if sp in rec and np.isfinite(rec[sp]["true"][ti])]
        has_other = len(present) > 0
        other_true = {sp: rec[sp]["true"][ti] for sp in present}
        gap = (ht[ti] - max(other_true.values())) if present else np.nan

        c_high     = ht[ti] >= HIGH_HUMAN
        c_human_ok = abs(ht[ti] - hp[ti]) <= ERR_WELL
        c_present  = has_other
        c_low      = has_other and all(v <= LOW_OTHER for v in other_true.values())
        c_other_ok = has_other and all(abs(rec[sp]["true"][ti] - rec[sp]["pred"][ti]) <= ERR_WELL for sp in present)
        c_gap      = has_other and (gap >= MIN_USAGE_GAP)

        # independent failures
        if not c_high:     fail["human_true < HIGH_HUMAN"] += 1
        if not c_human_ok: fail["human pred error > ERR_WELL"] += 1
        if not c_present:  fail["no other species aligned"] += 1
        if not c_low:      fail["some other species >= LOW_OTHER"] += 1
        if not c_other_ok: fail["some other species pred error > ERR_WELL"] += 1
        if not c_gap:      fail["gap < MIN_USAGE_GAP"] += 1

        # cumulative funnel (order matters)
        stages = [
            ("high human usage",      c_high),
            ("+ human well predicted", c_human_ok),
            ("+ other species present", c_present),
            ("+ others low usage",     c_low),
            ("+ others well predicted", c_other_ok),
            ("+ gap >= MIN_USAGE_GAP",  c_gap),
        ]
        ok = True
        for name, cond in stages:
            ok = ok and bool(cond)
            if ok:
                cum[name] += 1

print(f"Config: HIGH_HUMAN={HIGH_HUMAN}  LOW_OTHER={LOW_OTHER}  MIN_USAGE_GAP={MIN_USAGE_GAP}  ERR_WELL={ERR_WELL}")
print(f"{tot_conditions:,} (site, timepoint) conditions with finite human usage "
      f"across {len(sites_with_finite):,} sites\n")

print("Independent failures (each criterion alone, out of all conditions):")
order = ["human_true < HIGH_HUMAN", "human pred error > ERR_WELL", "no other species aligned",
         "some other species >= LOW_OTHER", "some other species pred error > ERR_WELL",
         "gap < MIN_USAGE_GAP"]
for k in order:
    v = fail.get(k, 0)
    print(f"  {v:7,} ({100*v/max(tot_conditions,1):5.1f}%)  {k}")

print("\nCumulative funnel (criteria applied in order):")
print(f"  {tot_conditions:7,} (100.0%)  start: finite human usage")
for name, _ in [("high human usage",0),("+ human well predicted",0),("+ other species present",0),
                ("+ others low usage",0),("+ others well predicted",0),("+ gap >= MIN_USAGE_GAP",0)]:
    v = cum.get(name, 0)
    print(f"  {v:7,} ({100*v/max(tot_conditions,1):5.1f}%)  {name}")


Config: HIGH_HUMAN=0.6  LOW_OTHER=0.4  MIN_USAGE_GAP=0.2  ERR_WELL=0.12
58,127 (site, timepoint) conditions with finite human usage across 4,887 sites

Independent failures (each criterion alone, out of all conditions):
   17,236 ( 29.7%)  human_true < HIGH_HUMAN
   19,254 ( 33.1%)  human pred error > ERR_WELL
   44,727 ( 76.9%)  no other species aligned
   57,009 ( 98.1%)  some other species >= LOW_OTHER
   50,453 ( 86.8%)  some other species pred error > ERR_WELL
   57,256 ( 98.5%)  gap < MIN_USAGE_GAP

Cumulative funnel (criteria applied in order):
   58,127 (100.0%)  start: finite human usage
   40,891 ( 70.3%)  high human usage
   28,067 ( 48.3%)  + human well predicted
    8,618 ( 14.8%)  + other species present
      226 (  0.4%)  + others low usage
       21 (  0.0%)  + others well predicted
       21 (  0.0%)  + gap >= MIN_USAGE_GAP


## 5 — Example sites: true vs predicted usage across species (FOCUS_TISSUE)

In [ ]:
N_PLOT = 10

def inspect_site(hsite):
    chrom, pos, _ = sx.parse_site(hsite)
    htps = sorted(result.loc[result["site"] == hsite, "timepoint"].astype(int).unique().tolist())
    frames, coords, notes = [], [], {}
    hf = load_all_tissues("human", [pos])
    if hf is not None and not hf.empty:
        frames.append(hf); coords.append(f"human {hsite}")
    for sp in OTHER_SPECIES:
        r = records[hsite].get(sp)
        if not r:
            notes[sp] = "no aligned splice site"; continue
        _, ap, _ = sx.parse_site(r["aln_site"])
        f = load_all_tissues(sp, [ap])
        if f is None or f.empty:
            notes[sp] = "no data"; continue
        frames.append(f); coords.append(f"{sp} {r['aln_site']}")
    if notes:
        print(f"{hsite}:", "; ".join(f"{k}: {v}" for k, v in notes.items()))
    if len(coords) < 2:
        return
    fig = plot_splice_site_predictions(
        pd.concat(frames, ignore_index=True), site_coords=coords,
        true_col="SSE_true", pred_col="SSE_pred",
        tissue_colors=TISSUE_COLORS, tissue_order=TISSUE_ORDER, tissue_subset=[FOCUS_TISSUE],
        n_cols=len(coords), split_by_tissue=False,
        highlight_timepoints=htps,
        title=f"{hsite} \u2014 human high, low elsewhere ({FOCUS_TISSUE}); "
              f"qualifying tp: {', '.join(map(str, htps))}",
    )
    if fig is not None:
        for ax in fig.axes: ax.set_ylim(-0.05, 1.05)
        fig.savefig(os.path.join(OUT_DIR, f"usage_{hsite.replace(':', '_')}.png"),
                    dpi=200, bbox_inches="tight")
        plt.show()

for hsite in result["site"].drop_duplicates().head(N_PLOT):
    inspect_site(hsite)